In [1]:
# Imports and paths

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

raw_csv_path = DATA_DIR / "synthetic_telecom_kpi_data.csv"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Raw CSV path:", raw_csv_path)

Project directory: D:\aws-kpi-anomaly-detection
Data directory: D:\aws-kpi-anomaly-detection\data
Raw CSV path: D:\aws-kpi-anomaly-detection\data\synthetic_telecom_kpi_data.csv


In [6]:
# Load the dataset
kpi_df = pd.read_csv(raw_csv_path, parse_dates=["timestamp"])

print("Dataset shape:", kpi_df.shape)
kpi_df.head()

Dataset shape: (11520, 13)


,timestamp,cell_id,rrc_setup_success_rate,handover_success_rate,call_drop_rate,throughput_mbps,prb_utilization,latency_ms,hour,day_of_week,is_weekend,anomaly_label,anomaly_type
0,2026-01-01 00:00:00,CELL_001,99.390610,99.213203,0.396134,81.411686,26.401183,24.030174,0,3,0,0,normal
1,2026-01-01 01:00:00,CELL_001,99.211772,98.523758,0.325324,83.998258,23.833997,18.963413,1,3,0,0,normal
2,2026-01-01 02:00:00,CELL_001,99.635269,98.277058,0.289934,69.678204,28.535084,18.768624,2,3,0,0,normal
3,2026-01-01 03:00:00,CELL_001,99.433403,98.799203,0.051910,77.341371,28.701212,26.548894,3,3,0,0,normal
4,2026-01-01 04:00:00,CELL_001,99.436659,98.493148,0.250085,68.468843,39.874322,27.480007,4,3,0,0,normal


In [8]:
# Select features for SageMaker. Important: we are not using anomaly_label for training. We keep it only for later evaluation.
feature_cols = [
    "rrc_setup_success_rate",
    "handover_success_rate",
    "call_drop_rate",
    "throughput_mbps",
    "prb_utilization",
    "latency_ms",
    "hour",
    "day_of_week",
    "is_weekend"
]

label_col = "anomaly_label"

print("Feature columns used for SageMaker:")
for col in feature_cols:
    print("-", col)

Feature columns used for SageMaker:
- rrc_setup_success_rate
- handover_success_rate
- call_drop_rate
- throughput_mbps
- prb_utilization
- latency_ms
- hour
- day_of_week
- is_weekend


In [10]:
# Time-based train/test split. 
kpi_df = kpi_df.sort_values(["timestamp", "cell_id"]).reset_index(drop=True)

split_time = kpi_df["timestamp"].quantile(0.70)

train_df = kpi_df[kpi_df["timestamp"] <= split_time].copy()
test_df = kpi_df[kpi_df["timestamp"] > split_time].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain anomaly percentage:")
print(train_df[label_col].value_counts(normalize=True) * 100)

print("\nTest anomaly percentage:")
print(test_df[label_col].value_counts(normalize=True) * 100)

Train shape: (8064, 13)
Test shape: (3456, 13)

Train anomaly percentage:
0    97.247024
1     2.752976
Name: anomaly_label, dtype: float64

Test anomaly percentage:
0    97.685185
1     2.314815
Name: anomaly_label, dtype: float64


In [12]:
# Scale the features. This creates normalized numerical data for SageMaker.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(train_df[feature_cols])
X_test_scaled = scaler.transform(test_df[feature_cols])

train_features_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
test_features_df = pd.DataFrame(X_test_scaled, columns=feature_cols)

train_features_df.head()


,rrc_setup_success_rate,handover_success_rate,call_drop_rate,throughput_mbps,prb_utilization,latency_ms,hour,day_of_week,is_weekend
0,0.841825,1.413490,-0.954549,1.116454,-1.307837,-0.959673,-1.661325,0.0,-0.632456
1,1.219704,0.757875,-1.489732,1.310485,-1.552790,-1.085200,-1.661325,0.0,-0.632456
2,0.627722,0.616980,-0.259956,0.691960,-0.852289,-1.519304,-1.661325,0.0,-0.632456
3,0.452111,-0.097382,-0.632510,1.022874,-1.286191,-1.294126,-1.661325,0.0,-0.632456
4,0.042392,0.454818,-0.839197,1.632300,-1.375533,-0.762210,-1.661325,0.0,-0.632456


In [14]:
# Save SageMaker-ready files locally.
train_features_path = DATA_DIR / "sagemaker_train_features.csv"
test_features_path = DATA_DIR / "sagemaker_test_features.csv"
test_labels_path = DATA_DIR / "sagemaker_test_labels.csv"
test_metadata_path = DATA_DIR / "sagemaker_test_metadata.csv"
scaler_path = OUTPUTS_DIR / "sagemaker_standard_scaler.joblib"

# Feature files for SageMaker training/inference
# Header=False because many SageMaker built-in algorithms expect plain numerical CSV input
train_features_df.to_csv(train_features_path, index=False, header=False)
test_features_df.to_csv(test_features_path, index=False, header=False)

# Labels for later evaluation
test_df[[label_col, "anomaly_type"]].to_csv(test_labels_path, index=False)

# Metadata for visualization and interpretation
test_df[["timestamp", "cell_id"] + feature_cols + [label_col, "anomaly_type"]].to_csv(
    test_metadata_path,
    index=False
)

# Save scaler locally
joblib.dump(scaler, scaler_path)

print("Saved files:")
print(train_features_path)
print(test_features_path)
print(test_labels_path)
print(test_metadata_path)
print(scaler_path)

Saved files:
D:\aws-kpi-anomaly-detection\data\sagemaker_train_features.csv
D:\aws-kpi-anomaly-detection\data\sagemaker_test_features.csv
D:\aws-kpi-anomaly-detection\data\sagemaker_test_labels.csv
D:\aws-kpi-anomaly-detection\data\sagemaker_test_metadata.csv
D:\aws-kpi-anomaly-detection\outputs\sagemaker_standard_scaler.joblib


In [16]:
# Verify the files
print("Train features shape:", train_features_df.shape)
print("Test features shape:", test_features_df.shape)
print("Test labels shape:", test_df[[label_col, "anomaly_type"]].shape)
print("Test metadata shape:", test_df[["timestamp", "cell_id"] + feature_cols + [label_col, "anomaly_type"]].shape)

Train features shape: (8064, 9)
Test features shape: (3456, 9)
Test labels shape: (3456, 2)
Test metadata shape: (3456, 13)
